Step 1 — Verify hardware, persistent storage and pinned source

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess

ROOT = Path("/workspace/B3IT_Figure1_Corrected")
REPO = ROOT / "authors_repo"
COMMIT = "ad2f69295797a5f03e7b480246080d91087d9007"
REPO_URL = (
    "https://github.com/timothee-chauvin/"
    "token-efficient-change-detection-llm-apis.git"
)

def run_command(arguments, cwd=None):
    result = subprocess.run(
        arguments,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )
    return result.stdout.strip()

# Verify persistent storage before creating project files.
mount_text = run_command([
    "findmnt", "-J", "-T", "/workspace",
    "-o", "TARGET,SOURCE,FSTYPE",
])
mount = json.loads(mount_text)["filesystems"][0]
assert mount["target"] == "/workspace" and mount["fstype"] != "overlay", (
    f"Persistent /workspace mount could not be verified: {mount}"
)
print("Workspace mount verified:", mount["fstype"])

gpu_report = run_command([
    "nvidia-smi",
    "--query-gpu=index,name,memory.total,memory.free,driver_version",
    "--format=csv,noheader",
])
print("\nGPU report:")
print(gpu_report)

memory_info = {}
for line in Path("/proc/meminfo").read_text().splitlines():
    key, value = line.split(":", 1)
    memory_info[key] = value.strip()

ram_total_gib = int(memory_info["MemTotal"].split()[0]) / 1024**2
ram_available_gib = int(memory_info["MemAvailable"].split()[0]) / 1024**2

disk_report = {}
for location in ("/workspace", "/"):
    usage = shutil.disk_usage(location)
    disk_report[location] = {
        "total_gib": usage.total / 1024**3,
        "free_gib": usage.free / 1024**3,
    }

print(f"\nSystem RAM: {ram_total_gib:.1f} GiB total")
print(f"Available RAM: {ram_available_gib:.1f} GiB")
for location, usage in disk_report.items():
    print(f"{location}: {usage['free_gib']:.1f} GiB free")

for folder in ("audit", "notebooks", "scripts", "models", "results", "cache"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

# Confirm actual write access.
write_test = ROOT / "audit" / "write_test.tmp"
write_test.write_text("persistent storage write test", encoding="utf-8")
assert write_test.read_text() == "persistent storage write test"
write_test.unlink()

if not REPO.exists():
    print("\nCloning authors' repository...", flush=True)
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO)],
        check=True,
    )
    subprocess.run(
        ["git", "checkout", "--detach", COMMIT],
        cwd=REPO,
        check=True,
    )

actual_commit = run_command(["git", "rev-parse", "HEAD"], cwd=REPO)
assert actual_commit == COMMIT, f"Unexpected commit: {actual_commit}"

source_changes = run_command(["git", "status", "--porcelain"], cwd=REPO)
assert not source_changes, (
    "The authors_repo contains changes. Share this output:\n" + source_changes
)

rules = {
    "repository": REPO_URL,
    "commit": COMMIT,
    "models": [
        "Qwen/Qwen2.5-0.5B-Instruct",
        "google/gemma-3-1b-it",
        "microsoft/Phi-4-mini-instruct",
    ],
    "workflow": {
        "one_cell_at_a_time": True,
        "estimate_time_each_step": True,
        "wait_for_user_output": True,
        "independent_model_notebooks": True,
        "fresh_inputs_and_results": True,
        "reuse_upstream_functions": True,
        "record_all_deviations": True,
        "resume_completed_stages": True,
    },
    "discovery": {
        "traffic": "Full upstream LMSYS loader and filters",
        "candidates": 20000,
        "queries_per_candidate": 3,
        "temperature": 0,
        "output_tokens": 1,
        "one_full_original_sampler_call": True,
    },
    "calibration": {
        "observations_per_discovered_input": 1000,
        "one_full_original_sampler_call": True,
        "selected_probes": 120,
        "selection": "Upstream eligibility and insertion order",
        "report_if_fewer_than_120": True,
    },
    "perturbations": {
        "lora_examples": 128,
        "lora_epochs": 1,
        "lora_batch_size": 1,
        "lora_optimizer": "SGD",
        "lora_learning_rate": 1e-6,
        "lora_rank": 8,
        "lora_alpha": 8,
        "lora_dropout": 0,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "training_data_and_sequence_handling": "Literal upstream behavior",
        "phi_fused_qkv": "Preserve original architecture",
        "noise_std": 2**-12,
        "magnitude_pruning_fraction": 2**-8,
        "random_pruning_fraction": 2**-8,
        "creation_dtype": "Upstream BF16 behavior",
        "checkpointing": "Only if needed, after equivalence check",
    },
    "evaluation": {
        "conditions_per_model": 5,
        "observations_per_probe_per_condition": 5000,
        "reference_samples": 50,
        "prompt_counts": [1, 3, 5, 10],
        "detection_sample_counts": [1, 3, 10, 20, 50],
        "repeated_tests": 1000,
        "per_variant_rows_per_model": 80,
        "summary_rows_per_model": 20,
        "scoring_cost_and_uncertainty": "Original repository functions",
        "combined_aggregation": "Equal mean of 12 model-variant AUCs per setting",
    },
}

rules_path = ROOT / "audit" / "experiment_rules.json"
if rules_path.exists():
    assert json.loads(rules_path.read_text()) == rules, (
        "Existing experiment rules differ. Share the error before continuing."
    )
else:
    rules_path.write_text(json.dumps(rules, indent=2), encoding="utf-8")

report = {
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "mount": mount,
    "gpu_report": gpu_report,
    "ram_total_gib": ram_total_gib,
    "ram_available_gib": ram_available_gib,
    "disk": disk_report,
    "commit": actual_commit,
    "native_bf16_calculation_verified": False,
    "training_memory_measured": False,
}
(ROOT / "audit" / "step01_preflight.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

deviation_log = ROOT / "audit" / "deviations.json"
if not deviation_log.exists():
    deviation_log.write_text("[]\n", encoding="utf-8")

print("\nStep 1 passed.")
print("Project:", ROOT)
print("Pinned commit:", actual_commit)
print("Experiment rules:", rules_path)
print("Next step: install locked dependencies and verify native BF16.")

Workspace mount verified: fuse

GPU report:
0, NVIDIA A40, 46068 MiB, 45490 MiB, 595.91.07

System RAM: 503.5 GiB total
Available RAM: 405.1 GiB
/workspace: 256320.4 GiB free
/: 50.0 GiB free

Cloning authors' repository...


Cloning into '/workspace/B3IT_Figure1_Corrected/authors_repo'...
HEAD is now at ad2f692 docs(demo): hold the last frame of the README recording for 10s (#2)



Step 1 passed.
Project: /workspace/B3IT_Figure1_Corrected
Pinned commit: ad2f69295797a5f03e7b480246080d91087d9007
Experiment rules: /workspace/B3IT_Figure1_Corrected/audit/experiment_rules.json
Next step: install locked dependencies and verify native BF16.


Step 2 — Install locked dependencies and test native BF16.

In [4]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

ROOT = Path("/workspace/B3IT_Figure1_Corrected")
PROJECT_DIR = ROOT / "authors_repo" / "track-llm-apis"
VENV = Path("/opt/b3it-corrected-venv")
PYTHON = str(VENV / "bin" / "python")
AUDIT = ROOT / "audit"

assert (PROJECT_DIR / "uv.lock").is_file(), "Run Step 1 first."
AUDIT.mkdir(parents=True, exist_ok=True)

# Install uv only if it is unavailable.
uv = shutil.which("uv")
if uv is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "uv"],
        check=True,
    )
    uv_command = [sys.executable, "-m", "uv"]
else:
    uv_command = [uv]

env = os.environ.copy()
env.update({
    "UV_PROJECT_ENVIRONMENT": str(VENV),
    "UV_PYTHON_INSTALL_DIR": "/opt/b3it-corrected-python",
    "MPLBACKEND": "Agg",
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
})

def run_logged(command, log_path):
    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            cwd=PROJECT_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()

    assert return_code == 0, (
        f"Command failed with exit code {return_code}. "
        f"Log saved at {log_path}"
    )

started = time.monotonic()
print("Installing dependencies from the pinned lockfile...", flush=True)

run_logged(
    [
        *uv_command,
        "sync",
        "--frozen",
        "--no-dev",
        "--python", "3.11",
        "--no-cache",
    ],
    AUDIT / "step02_installation.log",
)

check_code = r'''
import json
import sys
from importlib.metadata import version
from pathlib import Path

import torch
import track_llm_apis

assert sys.version_info[:2] == (3, 11), sys.version
assert torch.cuda.is_available(), "CUDA is unavailable."
assert torch.cuda.is_bf16_supported(including_emulation=False), (
    "Native BF16 support was not detected."
)

x = torch.ones((32, 32), device="cuda", dtype=torch.bfloat16)
y = x @ x
torch.cuda.synchronize()
assert y.dtype == torch.bfloat16 and y[0, 0].item() == 32
del x, y
torch.cuda.empty_cache()

record = {
    "python": sys.version,
    "python_executable": sys.executable,
    "packages": {
        name: version(name)
        for name in [
            "torch", "transformers", "vllm", "lightning",
            "peft", "datasets", "matplotlib"
        ]
    },
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "gpu_memory_gib": (
        torch.cuda.get_device_properties(0).total_memory / 1024**3
    ),
    "native_bf16_calculation_passed": True,
    "project_import": track_llm_apis.__file__,
    "model_inference_tested": False,
    "lora_training_tested": False,
}

Path(sys.argv[1]).write_text(
    json.dumps(record, indent=2), encoding="utf-8"
)
print(json.dumps(record, indent=2))
'''

run_logged(
    [
        PYTHON, "-u", "-c", check_code,
        str(AUDIT / "step02_environment.json"),
    ],
    AUDIT / "step02_environment_check.log",
)

packages = subprocess.check_output(
    [*uv_command, "pip", "freeze", "--python", PYTHON],
    cwd=PROJECT_DIR,
    env=env,
    text=True,
)
(AUDIT / "installed_packages.txt").write_text(
    packages, encoding="utf-8"
)

(AUDIT / "runtime_paths.json").write_text(
    json.dumps({
        "project_root": str(ROOT),
        "project_dir": str(PROJECT_DIR),
        "python": PYTHON,
        "environment_location": "container disk",
    }, indent=2),
    encoding="utf-8",
)

print(f"\nElapsed time: {(time.monotonic() - started) / 60:.1f} minutes")
print("Step 2 passed: locked dependencies installed; native BF16 verified.")
print("Experiment Python:", PYTHON)
print("Logs and package versions saved to:", AUDIT)

Installing dependencies from the pinned lockfile...
Using CPython 3.11.18 interpreter at: /usr/bin/python3.11
Creating virtual environment at: /opt/b3it-corrected-venv
Prepared 214 packages in 26.98s
Installed 214 packages in 1.48s
 + accelerate==1.12.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anthropic==0.77.1
 + anyio==4.12.1
 + apache-tvm-ffi==0.1.8.post2
 + astor==0.8.1
 + attrs==25.4.0
 + beartype==0.22.9
 + blake3==1.0.8
 + cachetools==7.0.0
 + cbor2==5.8.0
 + certifi==2026.1.4
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + choreographer==1.2.1
 + click==8.3.1
 + cloudpickle==3.1.2
 + compressed-tensors==0.13.0
 + contourpy==1.3.3
 + cryptography==46.0.4
 + cuda-bindings==13.1.1
 + cuda-pathfinder==1.3.3
 + cuda-python==13.1.1
 + cupy-cuda12x==13.6.0
 + cycler==0.12.1
 + datasets==4.5.0
 + deepspeed==0.18.5
 + depyf==0.20.0
 + diffusers==0.36.0
 + dill==0.4.0
 + diskcache==5.6.3
 + distro==1.9.0
 + dnsp

Using Python 3.11.18 environment at: /opt/b3it-corrected-venv


Step 3 — Verify Hugging Face access and record model/dataset revisions.

In [5]:
from pathlib import Path
from getpass import getpass
import os
import subprocess

ROOT = Path("/workspace/B3IT_Figure1_Corrected")
PYTHON = "/opt/b3it-corrected-venv/bin/python"

# Set these before starting the experiment subprocess.
os.environ["HF_HOME"] = str(ROOT / "cache" / "huggingface")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["MPLBACKEND"] = "Agg"

if not os.environ.get("HF_TOKEN"):
    token = getpass("Hugging Face read token: ").strip()
    assert token, "Enter your Hugging Face read token."
    os.environ["HF_TOKEN"] = token
    del token

check_access = r'''
import json
import os
from pathlib import Path

from huggingface_hub import HfApi, hf_hub_download

root = Path("/workspace/B3IT_Figure1_Corrected")
destination = root / "audit" / "hub_revisions.json"
token = os.environ["HF_TOKEN"]
api = HfApi(token=token)

models = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "google/gemma-3-1b-it",
    "microsoft/Phi-4-mini-instruct",
]
dataset = "lmsys/lmsys-chat-1m"

saved = json.loads(destination.read_text()) if destination.exists() else None
records = {}

for model in models:
    revision = saved["models"][model]["revision"] if saved else "main"
    info = api.model_info(model, revision=revision)
    assert info.sha, f"No revision returned for {model}"

    config_path = hf_hub_download(
        repo_id=model,
        filename="config.json",
        revision=info.sha,
        token=token,
    )
    config = json.loads(Path(config_path).read_text())

    records[model] = {
        "revision": info.sha,
        "model_type": config.get("model_type"),
        "architectures": config.get("architectures"),
        "declared_dtype": config.get("torch_dtype", config.get("dtype")),
        "max_position_embeddings": config.get("max_position_embeddings"),
    }
    print(f"Model access verified: {model}", flush=True)
    print(f"  Revision: {info.sha}", flush=True)

api.auth_check(repo_id=dataset, repo_type="dataset")
dataset_revision = saved["dataset"]["revision"] if saved else "main"
dataset_info = api.dataset_info(dataset, revision=dataset_revision)
assert dataset_info.sha, "No dataset revision returned."

manifest = {
    "authors_commit": "ad2f69295797a5f03e7b480246080d91087d9007",
    "models": records,
    "dataset": {
        "repo_id": dataset,
        "revision": dataset_info.sha,
    },
    "revision_note": (
        "Revisions resolved for this reproduction. "
        "The pinned authors' code does not specify historical Hub revisions."
    ),
}

if saved is not None:
    assert saved == manifest, "Saved revision manifest differs."
else:
    temporary = destination.with_suffix(".tmp")
    temporary.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    temporary.replace(destination)

print("LMSYS dataset access verified.", flush=True)
print("Dataset revision:", dataset_info.sha, flush=True)
print("Revision manifest:", destination, flush=True)
print("Step 3 passed.", flush=True)
'''

result = subprocess.run(
    [PYTHON, "-u", "-c", check_access],
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(result.stdout)
assert result.returncode == 0, "Access verification failed. Share the error."

Hugging Face read token:  ········


Model access verified: Qwen/Qwen2.5-0.5B-Instruct
  Revision: 7ae557604adf67be50417f59c2c2f167def9a775
Model access verified: google/gemma-3-1b-it
  Revision: dcc83ea841ab6100d6b47a070329e1ba4cf78752
Model access verified: microsoft/Phi-4-mini-instruct
  Revision: cfbefacb99257ffa30c83adab238a50856ac3083
LMSYS dataset access verified.
Dataset revision: 200748d9d3cddcc9d782887541057aca0b18c5da
Revision manifest: /workspace/B3IT_Figure1_Corrected/audit/hub_revisions.json
Step 3 passed.



Step 4 — Prepare the complete LMSYS dataset using the authors’ loader.

In [6]:
from pathlib import Path
import os
import subprocess
import time

ROOT = Path("/workspace/B3IT_Figure1_Corrected")
PROJECT_DIR = ROOT / "authors_repo" / "track-llm-apis"
PYTHON = "/opt/b3it-corrected-venv/bin/python"

for folder in ("scripts", "logs", "data/datasets"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

script = ROOT / "scripts" / "prepare_lmsys.py"

source = r'''
import json
from pathlib import Path

from track_llm_apis import util

root = Path("/workspace/B3IT_Figure1_Corrected")
revisions = json.loads((root / "audit/hub_revisions.json").read_text())
dataset_revision = revisions["dataset"]["revision"]
datasets_dir = root / "data/datasets"
report_path = root / "audit/lmsys_preparation.json"

# Preserve the original loader and filters.
# Supply only the explicitly recorded Hugging Face dataset revision.
original_load_dataset = util.load_dataset

def load_pinned_dataset(*args, **kwargs):
    dataset_name = args[0] if args else kwargs.get("path")
    if dataset_name == "lmsys/lmsys-chat-1m":
        assert kwargs.get("revision", dataset_revision) == dataset_revision
        kwargs["revision"] = dataset_revision
    return original_load_dataset(*args, **kwargs)

util.load_dataset = load_pinned_dataset

print("Preparing the full LMSYS dataset with upstream filters...", flush=True)
dataset = util.load_lmsys_chat_1m(
    gpt4_filter=True,
    redacted_filter=True,
    flagged_filter=True,
    first_turn_only=True,
    use_cache=True,
    datasets_dir=datasets_dir,
)

assert len(dataset) >= 128, "Too few filtered conversations."
assert dataset.column_names == ["conversation"]

for index, conversation in enumerate(dataset["conversation"]):
    assert len(conversation) == 2, f"Unexpected turn count at {index}"
    assert conversation[0]["role"] == "user"
    assert conversation[1]["role"] == "assistant"

record = {
    "dataset": "lmsys/lmsys-chat-1m",
    "revision": dataset_revision,
    "loader": "track_llm_apis.util.load_lmsys_chat_1m",
    "filters": {
        "gpt4_filter": True,
        "redacted_filter": True,
        "flagged_filter": True,
        "first_turn_only": True,
    },
    "full_dataset_processed": True,
    "filtered_conversations": len(dataset),
    "dataset_fingerprint": dataset._fingerprint,
    "datasets_dir": str(datasets_dir),
    "training_subset_selected": False,
    "token_length_filter_added": False,
}

if report_path.exists():
    saved = json.loads(report_path.read_text())
    assert saved == record, "Prepared dataset differs from its saved report."
else:
    temporary = report_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(record, indent=2), encoding="utf-8")
    temporary.replace(report_path)

print(json.dumps(record, indent=2), flush=True)
print("Step 4 passed: full upstream LMSYS preparation completed.", flush=True)
'''
script.write_text(source, encoding="utf-8")

env = os.environ.copy()
env["HF_HOME"] = str(ROOT / "cache/huggingface")
env["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
env["MPLBACKEND"] = "Agg"

started = time.monotonic()
log_path = ROOT / "logs/step04_lmsys_preparation.log"

with log_path.open("a", encoding="utf-8") as log:
    process = subprocess.Popen(
        [PYTHON, "-u", str(script)],
        cwd=PROJECT_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    exit_code = process.wait()

print(f"\nElapsed time: {(time.monotonic() - started) / 60:.1f} minutes")
assert exit_code == 0, f"Preparation failed. Share the error. Log: {log_path}"
print("Script saved:", script)
print("Log saved:", log_path)

Preparing the full LMSYS dataset with upstream filters...

Generating train split: 100%|██████████| 1000000/1000000 [00:13<00:00, 75625.10 examples/s]

Filter: 100%|██████████| 1000000/1000000 [00:00<00:00, 1164894.83 examples/s]

Map: 100%|██████████| 4570/4570 [00:07<00:00, 607.92 examples/s]

Filter: 100%|██████████| 4570/4570 [00:00<00:00, 9353.25 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 4502/4502 [00:00<00:00, 119080.02 examples/s]
{
  "dataset": "lmsys/lmsys-chat-1m",
  "revision": "200748d9d3cddcc9d782887541057aca0b18c5da",
  "loader": "track_llm_apis.util.load_lmsys_chat_1m",
  "filters": {
    "gpt4_filter": true,
    "redacted_filter": true,
    "flagged_filter": true,
    "first_turn_only": true
  },
  "full_dataset_processed": true,
  "filtered_conversations": 4502,
  "dataset_fingerprint": "3cd5574d57e1498f",
  "datasets_dir": "/workspace/B3IT_Figure1_Corrected/data/datasets",
  "training_subset_selected": false,
  "token_length_filter_added": false
}

Step 5 — Download the pinned Qwen model and tokenizer

In [7]:
import os
import subprocess
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
python = "/opt/b3it-corrected-venv/bin/python"

code = r'''
import json
import sys
from pathlib import Path

from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer

root = Path(sys.argv[1])
model = "Qwen/Qwen2.5-0.5B-Instruct"
manifest = json.loads((root / "audit/hub_revisions.json").read_text())
revision = manifest["models"][model]["revision"]

snapshot = Path(snapshot_download(
    repo_id=model,
    revision=revision,
    cache_dir=str(root / "cache/huggingface/hub"),
    allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],
))
assert snapshot.name == revision, f"Unexpected snapshot: {snapshot}"

index = snapshot / "model.safetensors.index.json"
if index.exists():
    shard_names = sorted(set(json.loads(index.read_text())["weight_map"].values()))
else:
    shard_names = ["model.safetensors"]

assert shard_names
for name in shard_names:
    shard = snapshot / name
    assert shard.is_file() and shard.stat().st_size > 0, f"Missing weights: {name}"

config = AutoConfig.from_pretrained(snapshot, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(snapshot, local_files_only=True)
assert config.model_type == "qwen2"
assert tokenizer.encode("Hello", add_special_tokens=False)

record = {
    "model": model,
    "revision": revision,
    "snapshot": str(snapshot),
    "weight_files": {name: (snapshot / name).stat().st_size for name in shard_names},
    "model_type": config.model_type,
    "tokenizer_class": type(tokenizer).__name__,
}
report = root / "audit/qwen_snapshot.json"
if report.exists():
    assert json.loads(report.read_text()) == record, "Existing Qwen snapshot record differs."
else:
    tmp = report.with_suffix(".tmp")
    tmp.write_text(json.dumps(record, indent=2) + "\n")
    tmp.replace(report)

print("Revision:", revision)
print("Snapshot:", snapshot)
print("Weights:", round(sum(record["weight_files"].values()) / 2**30, 2), "GiB")
print("Step 5 passed: pinned Qwen weights present and tokenizer loaded locally.")
'''

env = os.environ.copy()
env.update({
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
    "MPLBACKEND": "Agg",
})
subprocess.run([python, "-u", "-c", code, str(root)], env=env, check=True)

Fetching 7 files: 100%|██████████| 7/7 [00:04<00:00,  1.56it/s]


Revision: 7ae557604adf67be50417f59c2c2f167def9a775
Snapshot: /workspace/B3IT_Figure1_Corrected/cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775
Weights: 0.92 GiB
Step 5 passed: pinned Qwen weights present and tokenizer loaded locally.


CompletedProcess(args=['/opt/b3it-corrected-venv/bin/python', '-u', '-c', '\nimport json\nimport sys\nfrom pathlib import Path\n\nfrom huggingface_hub import snapshot_download\nfrom transformers import AutoConfig, AutoTokenizer\n\nroot = Path(sys.argv[1])\nmodel = "Qwen/Qwen2.5-0.5B-Instruct"\nmanifest = json.loads((root / "audit/hub_revisions.json").read_text())\nrevision = manifest["models"][model]["revision"]\n\nsnapshot = Path(snapshot_download(\n    repo_id=model,\n    revision=revision,\n    cache_dir=str(root / "cache/huggingface/hub"),\n    allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"],\n))\nassert snapshot.name == revision, f"Unexpected snapshot: {snapshot}"\n\nindex = snapshot / "model.safetensors.index.json"\nif index.exists():\n    shard_names = sorted(set(json.loads(index.read_text())["weight_map"].values()))\nelse:\n    shard_names = ["model.safetensors"]\n\nassert shard_names\nfor name in shard_names:\n    shard = snapshot / name\n    assert s

Step 6 runs Qwen border-input discovery

In [11]:
import os
import subprocess
import textwrap
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/discover_qwen.py"
log = root / "logs/step06_qwen_discovery_retry.log"
source = script.read_text(encoding="utf-8")

guard = 'if __name__ == "__main__":'
if guard not in source:
    marker = "root = Path(sys.argv[1])"
    assert source.count(marker) == 1, "Unexpected script layout. Share this output."
    imports, body = source.split(marker, 1)
    fixed = (
        imports + "def main():\n"
        + textwrap.indent(marker + body, "    ")
        + f"\n\n{guard}\n    main()\n"
    )
    compile(fixed, str(script), "exec")
    tmp = script.with_suffix(".py.tmp")
    tmp.write_text(fixed, encoding="utf-8")
    tmp.replace(script)

print("Worker import guard present. Starting Step 6.", flush=True)
env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
with log.open("w", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis",
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 6 exited with code {status}. Retry log: {log}")

Worker import guard present. Starting Step 6.
Starting upstream phase_1a on all 20,000 candidates.
2026-11-14 09:28:52,931 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 09:28:52,932 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
2026-11-14 09:28:53,040 - track-llm-apis - INFO - Loaded 4502 traffic prompts
2026-11-14 09:28:53,040 - track-llm-apis - INFO - Processing Qwen/Qwen2.5-0.5B-Instruct: 20000 inputs, 3 samples each, max_tokens=1
Enforce eager set, overriding optimization level to -O0
Allowing insecure serialization using pickle due to VLLM_ALLOW_INSECURE_SERIALIZATION=1
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=2234) 
Loading safetensors

Step 7 — Calibrate all 3,002 Qwen border inputs.

In [12]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/calibrate_qwen.py"
log = root / "logs/step07_qwen_calibration.log"

script.write_text(r'''
import hashlib
import json
import sys
from pathlib import Path
from unittest.mock import patch

from track_llm_apis.config import config
from track_llm_apis.sampling import bi_phase_1 as phase


def main():
    root = Path(sys.argv[1])
    model = "Qwen/Qwen2.5-0.5B-Instruct"
    snapshot_record = json.loads((root / "audit/qwen_snapshot.json").read_text())
    discovery = json.loads((root / "audit/step06_qwen_discovery.json").read_text())
    snapshot = Path(snapshot_record["snapshot"])
    assert model == discovery["model"] == snapshot_record["model"]
    assert snapshot.is_dir() and snapshot.name == discovery["revision"] == snapshot_record["revision"]
    assert config.data_dir == root / "data"

    bi = config.sampling.bi
    assert bi.phase_1_queries_per_bi == 1000 and bi.max_border_inputs == 120
    assert bi.temperature == 0 and bi.models[model] == 1
    assert (bi.batch_size, bi.per_batch_input_fraction) == (64, 0.95)

    phase_a = phase.BIPhase1aOutput.load_model(model)
    discovered = phase_a.get_border_inputs()
    assert len(discovered) == discovery["discovered_border_inputs"]
    assert phase_a.bi_config == bi.model_dump(mode="json")

    try:
        result = phase.BIPhase1bOutput.load_model(model)
    except FileNotFoundError:
        result = None

    if result is None:
        original_init = phase.init_vllm_bi

        def pinned_engine(name, device):
            assert name == model
            return original_init(str(snapshot), device)

        print(f"Calibrating {len(discovered):,} inputs × 1,000 observations.", flush=True)
        with patch.object(phase, "init_vllm_bi", pinned_engine):
            phase.phase_1b(device="cuda:0", model=model)
        result = phase.BIPhase1bOutput.load_model(model)
    else:
        print("Found saved calibration. Verifying every input.", flush=True)

    assert result.model_name == model and result.phase == "1b"
    assert result.bi_config == bi.model_dump(mode="json")
    assert set(result.results) == set(discovered), "Calibration input set differs from discovery."
    assert all(sum(counts.values()) == 1000 for counts in result.results.values())
    assert result.prompt_lengths.keys() == result.results.keys()

    eligible = result.get_border_inputs()
    selected = eligible[:bi.max_border_inputs]
    report = {
        "model": model,
        "revision": snapshot_record["revision"],
        "traffic_dataset_fingerprint": discovery["traffic_dataset_fingerprint"],
        "discovered_inputs": len(discovered),
        "calibration_observations": sum(sum(c.values()) for c in result.results.values()),
        "eligible_after_calibration": len(eligible),
        "selected_count": len(selected),
        "selection_rule": "First 120 returned by phase 1b get_border_inputs()",
        "prompts": selected,
        "prompt_lengths": {p: result.prompt_lengths[p] for p in selected},
        "prompts_sha256": hashlib.sha256(
            json.dumps(selected, ensure_ascii=False).encode("utf-8")
        ).hexdigest(),
        "upstream_result": str(result._get_path()),
    }
    destination = root / "results/qwen/selected_probes.json"
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert json.loads(destination.read_text()) == report, "Existing probe selection differs."
    else:
        tmp = destination.with_suffix(".tmp")
        tmp.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
        tmp.replace(destination)

    print(f"Calibration observations verified: {report['calibration_observations']:,}")
    print(f"Eligible after calibration: {len(eligible):,}")
    print(f"Selected probes: {len(selected)}")
    print("Saved:", destination)
    if len(selected) < 120:
        raise RuntimeError(f"Only {len(selected)} probes qualified. Actual results saved.")
    print("Step 7 passed: full Qwen calibration verified and 120 ordered probes saved.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "HF_HUB_ENABLE_HF_TRANSFER": "0",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis",
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 7 exited with code {status}. See {log}")

Calibrating 3,002 inputs × 1,000 observations.
2026-11-14 09:31:38,907 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 09:31:38,908 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
2026-11-14 09:31:39,016 - track-llm-apis - INFO - Loaded 4502 traffic prompts
2026-11-14 09:31:39,016 - track-llm-apis - INFO - Processing Qwen/Qwen2.5-0.5B-Instruct: 3002 inputs, 1000 samples each, max_tokens=1
Enforce eager set, overriding optimization level to -O0
Allowing insecure serialization using pickle due to VLLM_ALLOW_INSECURE_SERIALIZATION=1
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=2895) 
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?,

Step 8 — Check Qwen’s LoRA training data and memory requirements.

In [13]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
project = root / "authors_repo/track-llm-apis"
script = root / "scripts/check_qwen_lora_pool.py"

script.write_text(r'''
import json
import sys
from pathlib import Path

from transformers import AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.tinychange import TinyChangeConfig
from track_llm_apis.util import patch_chat_template


def main():
    root = Path(sys.argv[1])
    snapshot = json.loads((root / "audit/qwen_snapshot.json").read_text())
    cfg = TinyChangeConfig()
    pool = cfg.finetuning_dataset

    assert cfg.datasets_dir == root / "data/datasets"
    assert max(cfg.finetuning_samples) == 4096 and 128 in cfg.finetuning_samples
    assert len(pool) == 4096
    assert cfg.finetuning_batch_size == 1 and cfg.finetuning_epochs == 1
    assert cfg.finetuning_lr_scale == [1e-6]
    assert (cfg.lora_r, cfg.lora_alpha, cfg.lora_dropout) == (8, 8, 0.0)
    assert cfg.lora_target_modules == ["q_proj", "k_proj", "v_proj", "o_proj"]

    tokenizer = AutoTokenizer.from_pretrained(snapshot["snapshot"], local_files_only=True)
    patch_chat_template(tokenizer, config.chat_templates)

    lengths = []
    for start in range(0, len(pool), 32):
        conversations = pool[start:start + 32]["conversation"]
        encoded = tokenizer.apply_chat_template(
            conversations, tokenize=True, return_dict=True,
            padding=False, truncation=False,
        )
        lengths.extend(len(ids) for ids in encoded["input_ids"])

    longest = max(lengths)
    report = {
        "model": snapshot["model"],
        "revision": snapshot["revision"],
        "training_pool_size": len(pool),
        "training_pool_fingerprint": pool._fingerprint,
        "updates_for_selected_variant": 128,
        "configured_max_length": cfg.finetuning_max_length,
        "longest_untruncated_sequence": longest,
        "conversations_longer_than_configured_max": sum(
            n > cfg.finetuning_max_length for n in lengths
        ),
        "padded_token_positions_per_column": len(pool) * longest,
        "batch_size": cfg.finetuning_batch_size,
        "epochs": cfg.finetuning_epochs,
        "learning_rate": cfg.finetuning_lr_scale[0],
        "rank": cfg.lora_r,
        "alpha": cfg.lora_alpha,
        "dropout": cfg.lora_dropout,
        "target_modules": cfg.lora_target_modules,
    }
    path = root / "audit/step08_qwen_lora_pool.json"
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(report, indent=2) + "\n")
    tmp.replace(path)
    print(json.dumps(report, indent=2))
    print("Step 8 passed: upstream LoRA pool and sequence lengths verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=project, env=env, check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

2026-11-14 10:56:41,962 - matplotlib.font_manager - INFO - generated new fontManager
df: /root/.triton/autotune: No such file or directory
2026-11-14 10:56:47,858 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 10:56:47,859 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...


{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "revision": "7ae557604adf67be50417f59c2c2f167def9a775",
  "training_pool_size": 4096,
  "training_pool_fingerprint": "2c38c235042c87cc",
  "updates_for_selected_variant": 128,
  "configured_max_length": 1024,
  "longest_untruncated_sequence": 2477,
  "conversations_longer_than_configured_max": 19,
  "padded_token_positions_per_column": 10145792,
  "batch_size": 1,
  "epochs": 1,
  "learning_rate": 1e-06,
  "rank": 8,
  "alpha": 8,
  "dropout": 0.0,
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj"
  ]
}
Step 8 passed: upstream LoRA pool and sequence lengths verified.
Elapsed: 0.3 minutes


In [ ]:
Step 9 — Prepare and verify the Qwen LoRA training batch order

In [14]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/prepare_qwen_lora_data.py"

script.write_text(r'''
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
from transformers import AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.tinychange import FinetuningDataModule, TinyChangeConfig
from track_llm_apis.util import patch_chat_template


def main():
    root = Path(sys.argv[1])
    snapshot = json.loads((root / "audit/qwen_snapshot.json").read_text())
    measured = json.loads((root / "audit/step08_qwen_lora_pool.json").read_text())

    cfg = TinyChangeConfig()
    pool = cfg.finetuning_dataset
    assert cfg.datasets_dir == root / "data/datasets"
    assert len(pool) == measured["training_pool_size"] == 4096
    assert pool._fingerprint == measured["training_pool_fingerprint"]
    assert cfg.seed == 0 and cfg.finetuning_batch_size == 1

    tokenizer = AutoTokenizer.from_pretrained(snapshot["snapshot"], local_files_only=True)
    # The authors use the Hub model ID in the data module's cache directory name.
    tokenizer.name_or_path = snapshot["model"]
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    patch_chat_template(tokenizer, config.chat_templates)

    dm = FinetuningDataModule(
        config=cfg, n_samples=128,
        batch_size=cfg.finetuning_batch_size, tokenizer=tokenizer,
    )

    # TinyChange seeds NumPy before the data module creates its permutation.
    # Use a separate RNG to calculate the expected indices for verification.
    expected_indices = np.random.RandomState(cfg.seed).permutation(len(pool))[:128]
    np.random.seed(cfg.seed)
    dm.prepare_data()
    dm.setup("fit")

    selected = dm.tokenized_dataset
    assert len(selected) == 128
    padded_length = measured["longest_untruncated_sequence"]
    target_counts = []

    for position, pool_index in enumerate(expected_indices):
        raw = pool[int(pool_index)]["conversation"]
        expected = tokenizer.apply_chat_template(
            raw, tokenize=True, return_dict=True,
            return_assistant_tokens_mask=True, padding=False, truncation=False,
        )
        row = selected[position]
        ids = row["input_ids"].tolist()
        attention = row["attention_mask"].tolist()
        assistant = row["assistant_tokens_mask"].tolist()
        assert len(ids) == len(attention) == len(assistant) == padded_length, position
        assert row["labels"].tolist() == ids, position
        assert [v for v, keep in zip(ids, attention) if keep] == expected["input_ids"], position
        assert [v for v, keep in zip(assistant, attention) if keep] == expected["assistant_masks"], position
        assert all(a == 0 for a, keep in zip(assistant, attention) if not keep), position
        target_counts.append(sum(assistant[1:]))

    empty = [i for i, count in enumerate(target_counts) if count == 0]
    assert not empty, f"Examples with zero trainable assistant tokens: {empty}"

    indices = expected_indices.tolist()
    report = {
        "model": snapshot["model"],
        "revision": snapshot["revision"],
        "training_pool_fingerprint": pool._fingerprint,
        "seed": cfg.seed,
        "selected_examples": len(selected),
        "selected_pool_indices_in_training_order": indices,
        "indices_sha256": hashlib.sha256(json.dumps(indices).encode()).hexdigest(),
        "padded_tokens_per_example": padded_length,
        "minimum_trainable_assistant_tokens": min(target_counts),
        "data_module_cache": str(dm._get_disk_path()),
    }
    path = root / "audit/step09_qwen_lora_data.json"
    if path.exists():
        assert json.loads(path.read_text()) == report, "Existing training-data report differs."
    else:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(report, indent=2) + "\n")
        tmp.replace(path)

    print("Verified examples in training order:", len(selected))
    print("Padded tokens per example:", padded_length)
    print("Minimum trainable assistant tokens:", min(target_counts))
    print("Saved:", path)
    print("Step 9 passed: upstream Qwen LoRA data, order, and masks verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis", env=env, check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

2026-11-14 10:58:12,191 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 10:58:12,192 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
2026-11-14 10:58:12,472 - tinychange - INFO - Preprocessing finetuning dataset...
2026-11-14 10:58:12,597 - tinychange - INFO - Finetuning dataset does not exist at /workspace/B3IT_Figure1_Corrected/data/datasets/d49f919927c095fa/Qwen2fQwen2.5-0.5B-Instruct, preprocessing...
Saving the dataset (1/1 shards): 100%|██████████| 4096/4096 [00:00<00:00, 7760.04 examples/s]


Verified examples in training order: 128
Padded tokens per example: 2477
Minimum trainable assistant tokens: 2
Saved: /workspace/B3IT_Figure1_Corrected/audit/step09_qwen_lora_data.json
Step 9 passed: upstream Qwen LoRA data, order, and masks verified.
Elapsed: 0.3 minutes


In [ ]:
Step 10 — Train and save the Qwen LoRA variant

In [15]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/train_qwen_lora.py"

script.write_text(r'''
import json
import sys
import tempfile
from pathlib import Path
from unittest.mock import patch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from track_llm_apis import tinychange as tc
from track_llm_apis.config import config
from track_llm_apis.util import get_model_hash, patch_chat_template


def load_bf16(path):
    return AutoModelForCausalLM.from_pretrained(
        str(path), torch_dtype=torch.bfloat16, local_files_only=True,
    )


def main():
    root = Path(sys.argv[1])
    snapshot = json.loads((root / "audit/qwen_snapshot.json").read_text())
    data = json.loads((root / "audit/step09_qwen_lora_data.json").read_text())
    assert data["revision"] == snapshot["revision"]
    assert data["selected_examples"] == 128
    assert Path(data["data_module_cache"]).is_dir()

    dest = root / "results/qwen/checkpoints/lora_128"
    dest.parent.mkdir(parents=True, exist_ok=True)

    if dest.exists():
        saved = json.loads((dest / "COMPLETE.json").read_text())
        assert saved["revision"] == snapshot["revision"]
        assert saved["training_order_sha256"] == data["indices_sha256"]
        assert saved["optimizer_steps"] == 128
        assert get_model_hash(load_bf16(dest)) == saved["merged_hash"]
        print("Step 10 passed: existing Qwen LoRA checkpoint verified.", flush=True)
        print("Checkpoint:", dest)
        return

    cfg = tc.TinyChangeConfig(variants_device="cuda:0")
    assert cfg.finetuning_dataset._fingerprint == data["training_pool_fingerprint"]
    assert cfg.seed == data["seed"] == 0
    assert cfg.finetuning_lr_scale == [1e-6]
    assert cfg.finetuning_batch_size == cfg.finetuning_epochs == 1

    base = load_bf16(snapshot["snapshot"])
    assert base.device.type == "cpu" and base.dtype == torch.bfloat16
    tokenizer = AutoTokenizer.from_pretrained(snapshot["snapshot"], local_files_only=True)
    tokenizer.name_or_path = snapshot["model"]
    patch_chat_template(tokenizer, config.chat_templates)
    change = tc.TinyChange(base, tokenizer, cfg)

    record = {
        "model": snapshot["model"],
        "revision": snapshot["revision"],
        "training_order_sha256": data["indices_sha256"],
        "base_device": str(base.device),
        "base_dtype": str(base.dtype),
        "base_hash": change.model_hash,
        "training_device": cfg.variants_device,
        "seed": cfg.seed,
        "rank": cfg.lora_r,
        "alpha": cfg.lora_alpha,
        "dropout": cfg.lora_dropout,
        "target_modules": cfg.lora_target_modules,
    }

    # Observe the original trainer immediately before and after its fit call.
    original_fit = tc.pl.Trainer.fit

    def observed_fit(trainer, model, datamodule):
        assert Path(datamodule._get_disk_path()) == Path(data["data_module_cache"])
        assert not model.model.is_gradient_checkpointing
        trainable = {
            name: str(p.dtype)
            for name, p in model.model.named_parameters() if p.requires_grad
        }
        assert trainable and all("lora_" in name for name in trainable)
        record["trainable_parameters"] = trainable
        record["gradient_checkpointing"] = False
        print("Trainable adapter tensors:", len(trainable), flush=True)
        result = original_fit(trainer, model=model, datamodule=datamodule)
        record["optimizer_steps"] = int(trainer.global_step)
        assert record["optimizer_steps"] == 128
        return result

    torch.cuda.reset_peak_memory_stats()
    with patch.object(tc.pl.Trainer, "fit", observed_fit):
        variant = change.finetune(
            lr=cfg.finetuning_lr_scale[0], n_samples=128,
            epochs=cfg.finetuning_epochs, batch_size=cfg.finetuning_batch_size,
            use_lora=True,
        )

    record.update({
        "description": variant.description,
        "merged_dtype": str(variant.model.dtype),
        "merged_hash": variant.model_hash,
        "merged_weights_changed": variant.model_hash != change.model_hash,
        "peak_torch_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    })
    assert variant.model.dtype == torch.bfloat16

    # Publish the final directory only after reloading and verifying its weights.
    with tempfile.TemporaryDirectory(dir=dest.parent, prefix=".qwen-lora-") as tmp:
        staged = Path(tmp) / "checkpoint"
        variant.model.save_pretrained(staged, safe_serialization=True)
        tokenizer.save_pretrained(staged)
        assert get_model_hash(load_bf16(staged)) == record["merged_hash"]
        (staged / "COMPLETE.json").write_text(json.dumps(record, indent=2) + "\n")
        staged.rename(dest)

    for key in ("optimizer_steps", "merged_dtype", "merged_weights_changed",
                "peak_torch_allocated_gib"):
        print(f"{key}: {record[key]}")
    print("Checkpoint:", dest)
    print("Step 10 passed: Qwen LoRA trained, merged, saved, and reload-verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
}
log = root / "logs/step10_qwen_lora.log"
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis", env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 10 exited with code {status}. See {log}")

2026-11-14 10:59:49,544 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 10:59:49,545 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
`torch_dtype` is deprecated! Use `dtype` instead!
Seed set to 0
2026-11-14 11:00:03,475 - track-llm-apis - INFO - Copying model to cuda:0 with dtype torch.bfloat16...
/opt/b3it-corrected-venv/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /workspace/B3IT_Figure1_Corrected/scripts/train_qwen ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which lo

RuntimeError: Step 10 exited with code 1. See /workspace/B3IT_Figure1_Corrected/logs/step10_qwen_lora.log

In [16]:
import os
import subprocess
import time
from pathlib import Path
from textwrap import dedent, indent

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/train_qwen_lora.py"
source = script.read_text(encoding="utf-8")

old = '        assert get_model_hash(load_bf16(staged)) == record["merged_hash"]'
new = indent(dedent('''
    reloaded = load_bf16(staged)
    original_state = variant.model.state_dict()
    loaded_state = reloaded.state_dict()
    assert original_state.keys() == loaded_state.keys(), "Checkpoint tensor names differ"

    for name, tensor in original_state.items():
        actual = loaded_state[name]
        assert tensor.dtype == actual.dtype, f"Reload dtype mismatch: {name}"
        assert tensor.shape == actual.shape, f"Reload shape mismatch: {name}"
        assert torch.equal(tensor.detach().cpu(), actual), f"Reload value mismatch: {name}"

    before_buffers = dict(variant.model.named_buffers())
    after_buffers = dict(reloaded.named_buffers())
    buffer_differences = []
    for name in sorted((before_buffers.keys() | after_buffers.keys()) - original_state.keys()):
        before = before_buffers.get(name)
        after = after_buffers.get(name)
        if (before is None or after is None or before.dtype != after.dtype
                or before.shape != after.shape or not torch.equal(before.cpu(), after)):
            buffer_differences.append(name)

    base_params = dict(base.named_parameters())
    loaded_params = dict(reloaded.named_parameters())
    assert base_params.keys() == loaded_params.keys(), "Parameter layout differs from baseline"
    changed = 0
    for name, original in base_params.items():
        actual = loaded_params[name]
        assert original.dtype == actual.dtype == torch.bfloat16, name
        assert original.shape == actual.shape, name
        changed += int(torch.count_nonzero(original.detach() != actual.detach()).item())

    record["reloaded_hash"] = get_model_hash(reloaded)
    record["verified_checkpoint_tensors"] = len(loaded_state)
    record["nonpersistent_buffer_differences"] = buffer_differences
    record["changed_bf16_parameter_values"] = changed
    record["merged_weights_changed"] = changed > 0
    print("Verified checkpoint tensors:", len(loaded_state), flush=True)
    print("Non-persistent buffer differences:", buffer_differences, flush=True)
    print("Changed BF16 parameter values:", changed, flush=True)
''').strip(), "        ")

if old in source:
    assert source.count(old) == 1
    source = source.replace(old, new)
    old_resume = 'assert get_model_hash(load_bf16(dest)) == saved["merged_hash"]'
    assert source.count(old_resume) == 1
    source = source.replace(
        old_resume,
        'assert get_model_hash(load_bf16(dest)) == saved["reloaded_hash"]',
    )
    compile(source, str(script), "exec")
    tmp = script.with_suffix(".py.tmp")
    tmp.write_text(source, encoding="utf-8")
    tmp.replace(script)
else:
    assert 'record["verified_checkpoint_tensors"]' in source, "Unexpected Step 10 script"

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
}
log = root / "logs/step10_qwen_lora_retry.log"
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis", env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 10 retry exited with code {status}. See {log}")

2026-11-14 11:11:04,950 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 11:11:04,951 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
`torch_dtype` is deprecated! Use `dtype` instead!
Seed set to 0
2026-11-14 11:11:17,457 - track-llm-apis - INFO - Copying model to cuda:0 with dtype torch.bfloat16...
/opt/b3it-corrected-venv/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /workspace/B3IT_Figure1_Corrected/scripts/train_qwen ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which lo

In [ ]:
Step 11 — Create Qwen’s three remaining variants.

In [17]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/create_qwen_weight_variants.py"

script.write_text(r'''
import gc
import json
import sys
import tempfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from track_llm_apis import tinychange as tc
from track_llm_apis.util import get_model_hash, patch_chat_template
from track_llm_apis.config import config


def load_bf16(path):
    return AutoModelForCausalLM.from_pretrained(
        str(path), torch_dtype=torch.bfloat16, local_files_only=True
    )


def verify_saved_weights(model, reloaded):
    expected = model.state_dict()
    actual = reloaded.state_dict()
    assert expected.keys() == actual.keys(), "Checkpoint tensor names differ"
    for name, before in expected.items():
        after = actual[name]
        assert before.dtype == after.dtype, f"Reload dtype mismatch: {name}"
        assert before.shape == after.shape, f"Reload shape mismatch: {name}"
        assert torch.equal(before.detach().cpu(), after), f"Reload value mismatch: {name}"
    return len(actual)


def main():
    root = Path(sys.argv[1])
    snapshot = json.loads((root / "audit/qwen_snapshot.json").read_text())
    checkpoints = root / "results/qwen/checkpoints"
    lora = json.loads((checkpoints / "lora_128/COMPLETE.json").read_text())
    assert snapshot["revision"] == lora["revision"]

    base = load_bf16(snapshot["snapshot"])
    baseline_hash = get_model_hash(base)
    assert baseline_hash == lora["base_hash"], "LoRA and Step 11 baselines differ"
    base_params = dict(base.named_parameters())
    tokenizer = AutoTokenizer.from_pretrained(snapshot["snapshot"], local_files_only=True)
    tokenizer.name_or_path = snapshot["model"]
    patch_chat_template(tokenizer, config.chat_templates)

    cfg = tc.TinyChangeConfig(variants_device="cuda:0")
    settings = [
        ("noise", {"type": "random_noise", "scale": 2**-12}),
        ("magnitude", {"type": "weight_pruning", "method": "magnitude", "scale": 2**-8}),
        ("random", {"type": "weight_pruning", "method": "random", "scale": 2**-8}),
    ]
    assert cfg.seed == 0

    for name, description in settings:
        dest = checkpoints / name
        if dest.exists():
            saved = json.loads((dest / "COMPLETE.json").read_text())
            assert saved["base_hash"] == baseline_hash
            assert saved["revision"] == snapshot["revision"]
            assert saved["description"] == description
            assert saved["seed"] == cfg.seed
            assert get_model_hash(load_bf16(dest)) == saved["reloaded_hash"]
            print(f"{name}: existing checkpoint verified.", flush=True)
            continue

        print(f"Creating {name} from the original Qwen model...", flush=True)
        change = tc.TinyChange(base, tokenizer, cfg)
        if name == "noise":
            variant = change.random_noise(scale=description["scale"])
        else:
            variant = change.weight_pruning(scale=description["scale"], method=name)

        assert variant.description == description
        assert get_model_hash(base) == baseline_hash, "Original weights changed"
        with tempfile.TemporaryDirectory(dir=checkpoints, prefix=f".{name}-") as tmp:
            staged = Path(tmp) / "checkpoint"
            variant.model.save_pretrained(staged, safe_serialization=True)
            tokenizer.save_pretrained(staged)
            reloaded = load_bf16(staged)
            verified = verify_saved_weights(variant.model, reloaded)

            params = dict(reloaded.named_parameters())
            assert params.keys() == base_params.keys()
            changed = 0
            for key, original in base_params.items():
                actual = params[key].detach()
                assert actual.dtype == original.dtype == torch.bfloat16, key
                assert actual.shape == original.shape, key
                count = torch.count_nonzero(actual != original.detach()).item()
                if name != "noise" and count:
                    assert "mlp" in key and key.endswith(".weight"), key
                changed += int(count)

            report = {
                "model": snapshot["model"],
                "revision": snapshot["revision"],
                "description": description,
                "seed": cfg.seed,
                "rng_policy": "TinyChange initialized independently for each variant",
                "base_device": str(base.device),
                "creation_device": cfg.variants_device,
                "creation_dtype": str(variant.model.dtype),
                "base_hash": baseline_hash,
                "upstream_variant_hash": variant.model_hash,
                "reloaded_hash": get_model_hash(reloaded),
                "verified_checkpoint_tensors": verified,
                "changed_bf16_parameter_values": changed,
            }
            (staged / "COMPLETE.json").write_text(json.dumps(report, indent=2) + "\n")
            staged.rename(dest)

        print(f"{name}: {changed:,} BF16 parameter values changed; "
              f"{verified} checkpoint tensors verified.", flush=True)
        del params, actual, reloaded, variant, change
        gc.collect()
        torch.cuda.empty_cache()

    print("Step 11 passed: all three Qwen weight-perturbation checkpoints verified.")
    print("Checkpoint directory:", checkpoints)


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
    "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
}
log = root / "logs/step11_qwen_weight_variants.log"
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis", env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 11 exited with code {status}. See {log}")

`torch_dtype` is deprecated! Use `dtype` instead!
2026-11-14 11:20:40,481 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 11:20:40,482 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
Creating noise from the original Qwen model...
Seed set to 0
2026-11-14 11:20:46,078 - track-llm-apis - INFO - Copying model to cuda:0 with dtype torch.bfloat16...
noise: 430,398,704 BF16 parameter values changed; 291 checkpoint tensors verified.
Creating magnitude from the original Qwen model...
Seed set to 0
2026-11-14 11:21:28,674 - track-llm-apis - INFO - Copying model to cuda:0 with dtype torch.bfloat16...
magnitude: 1,225,728 BF16 parameter values changed; 291 checkpoint tensors verified.
Creating random from the original Qwen model...
Seed set to 0
2026-11-14 11:22:05,160 - track-llm-apis - INFO - Copying model to cuda:0 with dtype torch.bfloat16...
random: 1,225,728 BF16 par

Step 12 — Verify and save the Qwen evaluation manifest

In [18]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/prepare_qwen_evaluation.py"

script.write_text(r'''
import hashlib
import json
import sys
from pathlib import Path

from transformers import AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.sampling.bi_phase_1 import BIPhase1bOutput


def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False).encode()).hexdigest()


def encode_all(path, texts):
    tokenizer = AutoTokenizer.from_pretrained(str(path), local_files_only=True)
    return tokenizer(texts, padding=False, truncation=False)["input_ids"]


def main():
    root = Path(sys.argv[1])
    results = root / "results/qwen"
    snap = json.loads((root / "audit/qwen_snapshot.json").read_text())
    selection = json.loads((results / "selected_probes.json").read_text())
    prep = json.loads((root / "audit/lmsys_preparation.json").read_text())
    model = snap["model"]

    assert config.data_dir == root / "data"
    assert selection["model"] == model and selection["revision"] == snap["revision"]
    bi = config.sampling.bi
    lp = config.sampling.logprob

    calibration = BIPhase1bOutput.load_model(model, temperature=bi.temperature)
    probes = calibration.get_border_inputs()[:bi.max_border_inputs]
    assert probes == selection["prompts"] and len(set(probes)) == len(probes) == 120
    assert digest(probes) == selection["prompts_sha256"]

    traffic = lp.other_prompts
    assert lp.other_prompts_dataset._fingerprint == prep["dataset_fingerprint"]
    assert len(traffic) == prep["filtered_conversations"] == 4502
    assert (bi.temperature, bi.models[model], lp.batch_size) == (0, 1, 64)
    assert bi.per_batch_input_fraction == 0.95
    assert config.sampling.n_samples == 5000
    assert bi.reference_samples_per_prompt == 50

    texts = list(dict.fromkeys(probes + traffic))
    baseline = encode_all(snap["snapshot"], texts)
    conditions = [{
        "id": "unchanged", "model_path": snap["snapshot"],
        "description": {"type": "unchanged"},
    }]
    expected = {
        "lora_128": {"type": "finetune", "lr": 1e-6, "n_samples": 128, "lora": True},
        "noise": {"type": "random_noise", "scale": 2**-12},
        "magnitude": {"type": "weight_pruning", "method": "magnitude", "scale": 2**-8},
        "random": {"type": "weight_pruning", "method": "random", "scale": 2**-8},
    }
    base_hashes = set()
    for name, description in expected.items():
        path = results / "checkpoints" / name
        report = json.loads((path / "COMPLETE.json").read_text())
        assert report["model"] == model and report["revision"] == snap["revision"], name
        assert report["description"] == description, name
        assert report["changed_bf16_parameter_values"] > 0, name
        assert encode_all(path, texts) == baseline, f"{name}: tokenizer mismatch"
        base_hashes.add(report["base_hash"])
        conditions.append({
            "id": name, "model_path": str(path), "description": description,
            "checkpoint_reloaded_hash": report["reloaded_hash"],
        })
        print(f"{name}: variant metadata and evaluation token IDs verified.", flush=True)
    assert len(base_hashes) == 1, "Checkpoint baseline hashes differ"

    manifest = {
        "model": model, "revision": snap["revision"],
        "probes": probes, "probes_sha256": digest(probes),
        "traffic_count": len(traffic), "traffic_sha256": digest(traffic),
        "traffic_dataset_fingerprint": prep["dataset_fingerprint"],
        "conditions": conditions, "base_hash": base_hashes.pop(),
        "temperature": bi.temperature, "max_tokens": bi.models[model],
        "batch_size": lp.batch_size, "input_fraction": bi.per_batch_input_fraction,
        "observations_per_probe_per_condition": config.sampling.n_samples,
        "reference_samples_per_prompt": bi.reference_samples_per_prompt,
        "planned_target_observations": len(probes) * len(conditions) * config.sampling.n_samples,
    }
    assert manifest["planned_target_observations"] == 3_000_000
    path = results / "evaluation_manifest.json"
    if path.exists():
        assert json.loads(path.read_text()) == manifest, "Existing evaluation manifest differs"
    else:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
        tmp.replace(path)

    print("Saved:", path)
    print("Step 12 passed: 120 probes × 5 conditions × 5,000 fresh observations planned.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis", env=env, check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

2026-11-14 11:28:15,165 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 11:28:15,166 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
Traceback (most recent call last):
  File "/workspace/B3IT_Figure1_Corrected/scripts/prepare_qwen_evaluation.py", line 101, in <module>
    main()
  File "/workspace/B3IT_Figure1_Corrected/scripts/prepare_qwen_evaluation.py", line 40, in main
    assert lp.other_prompts_dataset._fingerprint == prep["dataset_fingerprint"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError


CalledProcessError: Command '['/opt/b3it-corrected-venv/bin/python', '-u', '/workspace/B3IT_Figure1_Corrected/scripts/prepare_qwen_evaluation.py', '/workspace/B3IT_Figure1_Corrected']' returned non-zero exit status 1.

In [19]:
import os
import subprocess
import time
from pathlib import Path
from textwrap import dedent, indent

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/prepare_qwen_evaluation.py"
source = script.read_text(encoding="utf-8")

old = '    assert lp.other_prompts_dataset._fingerprint == prep["dataset_fingerprint"]'
new = indent(dedent('''
    from datasets import load_from_disk
    from track_llm_apis.util import slugify

    cache = (
        Path(prep["datasets_dir"])
        / slugify(prep["dataset"], hash_length=0)
    ).resolve()
    state = json.loads((cache / "state.json").read_text())
    stored_fp = state["_fingerprint"]
    print("Step 4 fingerprint:", prep["dataset_fingerprint"], flush=True)
    print("On-disk fingerprint:", stored_fp, flush=True)
    print("Reloaded fingerprint:", lp.other_prompts_dataset._fingerprint, flush=True)
    assert stored_fp == prep["dataset_fingerprint"], "On-disk provenance differs from Step 4"

    expected_files = [(cache / f["filename"]).resolve() for f in state["_data_files"]]
    opened_files = [Path(f["filename"]).resolve() for f in lp.other_prompts_dataset.cache_files]
    assert expected_files and opened_files == expected_files, "Evaluation opened different cache files"

    reference = load_from_disk(str(cache))
    expected_traffic = [row["conversation"][0]["content"] for row in reference]
    assert traffic == expected_traffic, "Evaluation traffic content or order differs"
    print(f"Verified {len(traffic):,} ordered traffic prompts from the Step 4 cache.", flush=True)
''').strip(), "    ")

if old in source:
    assert source.count(old) == 1
    source = source.replace(old, new)
    compile(source, str(script), "exec")
    tmp = script.with_suffix(".py.tmp")
    tmp.write_text(source, encoding="utf-8")
    tmp.replace(script)
else:
    assert "On-disk provenance differs from Step 4" in source, "Unexpected Step 12 script"

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis", env=env, check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

2026-11-14 11:29:25,708 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 11:29:25,709 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...


Step 4 fingerprint: 3cd5574d57e1498f
On-disk fingerprint: 3cd5574d57e1498f
Reloaded fingerprint: 1ef91ba0c58e4259
Verified 4,502 ordered traffic prompts from the Step 4 cache.


The tokenizer you are loading from '/workspace/B3IT_Figure1_Corrected/results/qwen/checkpoints/lora_128' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


lora_128: variant metadata and evaluation token IDs verified.


The tokenizer you are loading from '/workspace/B3IT_Figure1_Corrected/results/qwen/checkpoints/noise' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


noise: variant metadata and evaluation token IDs verified.


The tokenizer you are loading from '/workspace/B3IT_Figure1_Corrected/results/qwen/checkpoints/magnitude' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


magnitude: variant metadata and evaluation token IDs verified.


The tokenizer you are loading from '/workspace/B3IT_Figure1_Corrected/results/qwen/checkpoints/random' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


random: variant metadata and evaluation token IDs verified.
Saved: /workspace/B3IT_Figure1_Corrected/results/qwen/evaluation_manifest.json
Step 12 passed: 120 probes × 5 conditions × 5,000 fresh observations planned.
Elapsed: 0.2 minutes


In [ ]:
Step 13 — Collect the unchanged Qwen baseline

In [20]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/collect_qwen_baseline.py"

script.write_text(r'''
import hashlib
import json
import random
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.sampling.vllm_sampling import (
    cleanup_vllm, init_vllm, load_model_to_vllm, vllm_inference_bi,
)
from track_llm_apis.util import patch_chat_template


def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False).encode()).hexdigest()


def verify(record, manifest, manifest_hash):
    probes = manifest["probes"]
    assert record["manifest_sha256"] == manifest_hash
    assert record["condition"] == "unchanged"
    assert record["output_tokens"] == 1
    assert set(record["counts"]) == set(record["prompt_lengths"]) == set(probes)
    for p in probes:
        counts = record["counts"][p]
        assert all(type(n) is int and n > 0 for n in counts.values()), repr(p)
        assert sum(counts.values()) == 5000, repr(p)
        assert record["prompt_lengths"][p] > 0
    version, state, gauss = record["python_random_state_after_sampling"]
    random.Random().setstate((version, tuple(state), gauss))


def main():
    root = Path(sys.argv[1])
    results = root / "results/qwen"
    manifest_file = results / "evaluation_manifest.json"
    manifest_bytes = manifest_file.read_bytes()
    manifest = json.loads(manifest_bytes)
    manifest_hash = hashlib.sha256(manifest_bytes).hexdigest()
    out = results / "observations/unchanged.json"
    out.parent.mkdir(parents=True, exist_ok=True)

    if out.exists():
        verify(json.loads(out.read_text()), manifest, manifest_hash)
        print("Step 13 passed: saved baseline reverified — 600,000 observations.")
        print("Saved:", out)
        return

    probes = manifest["probes"]
    n = manifest["observations_per_probe_per_condition"]
    assert len(set(probes)) == len(probes) == 120 and n == 5000
    assert digest(probes) == manifest["probes_sha256"]
    assert config.data_dir == root / "data"

    lp = config.sampling.logprob
    bi = config.sampling.bi
    traffic = lp.other_prompts
    assert digest(traffic) == manifest["traffic_sha256"]
    assert (lp.batch_size, bi.per_batch_input_fraction) == (64, 0.95)
    assert bi.temperature == manifest["temperature"] == 0
    assert bi.models[manifest["model"]] == manifest["max_tokens"] == 1

    baseline = next(c for c in manifest["conditions"] if c["id"] == "unchanged")
    path = Path(baseline["model_path"])
    assert path.name == manifest["revision"]
    tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
    patch_chat_template(tokenizer, config.chat_templates)
    model = AutoModelForCausalLM.from_pretrained(
        path, torch_dtype=torch.bfloat16, local_files_only=True,
    ).to("cuda:0")

    # Upstream evaluation initializes the engine, then loads condition weights via IPC.
    engine = init_vllm(model, tokenizer, "cuda:0")
    try:
        load_model_to_vllm(engine, model)
        actual_dtype = engine.llm_engine.vllm_config.model_config.dtype
        assert actual_dtype == torch.bfloat16, f"Evaluation engine dtype: {actual_dtype}"

        # Check the engine tokenizer as well as the checkpoint tokenizers from Step 12.
        engine_tokenizer = engine.get_tokenizer()
        for text in dict.fromkeys(probes + traffic):
            assert engine_tokenizer.encode(text) == tokenizer.encode(text), (
                "Evaluation engine tokenizer differs from the pinned tokenizer."
            )

        random.seed(config.seed)
        print("Collecting unchanged Qwen: 120 probes × 5,000 observations.", flush=True)
        sampled = vllm_inference_bi(
            llm=engine, inputs=probes, traffic_prompts=traffic, n_samples=n,
            temperature=bi.temperature, input_fraction=bi.per_batch_input_fraction,
            batch_size=lp.batch_size, max_tokens=manifest["max_tokens"],
        )
        record = {
            "model": manifest["model"],
            "revision": manifest["revision"],
            "condition": "unchanged",
            "description": baseline["description"],
            "manifest_sha256": manifest_hash,
            "inference_dtype": str(actual_dtype),
            "sampling_seed": config.seed,
            "counts": sampled.counts,
            "prompt_lengths": sampled.prompt_lengths,
            "output_tokens": sampled.output_tokens,
            "python_random_state_after_sampling": random.getstate(),
        }
        verify(record, manifest, manifest_hash)
        tmp = out.with_suffix(".tmp")
        tmp.write_text(json.dumps(record, ensure_ascii=False))
        verify(json.loads(tmp.read_text()), manifest, manifest_hash)
        tmp.replace(out)
    finally:
        cleanup_vllm(engine)

    print("Step 13 passed: 120 unchanged Qwen probes × 5,000 observations verified.")
    print("Total fresh baseline observations: 600,000")
    print("Saved:", out)


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
log = root / "logs/step13_qwen_baseline.log"
started = time.monotonic()
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
        cwd=root / "authors_repo/track-llm-apis", env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 13 exited with code {status}. See {log}")

2026-11-14 11:31:11,035 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 11:31:11,036 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
`torch_dtype` is deprecated! Use `dtype` instead!
Enforce eager set, overriding optimization level to -O0
The tokenizer you are loading from '/tmp/tmp01463945' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Allowing insecure serialization using pickle due to VLLM_ALLOW_INSECURE_SERIALIZATION=1
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information.

Step 14 — Collect the four changed Qwen conditions.

In [21]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/collect_qwen_changed.py"

script.write_text(r'''
import hashlib
import json
import random
import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from track_llm_apis.config import config
from track_llm_apis.sampling.vllm_sampling import (
    cleanup_vllm, init_vllm, load_model_to_vllm, vllm_inference_bi,
)
from track_llm_apis.util import get_model_hash, patch_chat_template
from collect_qwen_baseline import digest, verify as verify_baseline


def restore_rng(saved):
    version, state, gauss = saved
    random.setstate((version, tuple(state), gauss))


def verify_changed(record, condition, manifest, manifest_hash, start_hash, lengths):
    assert record["manifest_sha256"] == manifest_hash
    assert record["condition"] == condition["id"]
    assert record["description"] == condition["description"]
    assert record["checkpoint_hash"] == condition["checkpoint_reloaded_hash"]
    assert record["start_rng_sha256"] == start_hash
    assert record["inference_dtype"] == "torch.bfloat16"
    assert record["output_tokens"] == 1
    assert record["prompt_lengths"] == lengths
    assert set(record["counts"]) == set(manifest["probes"])
    for prompt, counts in record["counts"].items():
        assert all(type(n) is int and n > 0 for n in counts.values()), repr(prompt)
        assert sum(counts.values()) == 5000, repr(prompt)
    version, state, gauss = record["python_random_state_after_sampling"]
    random.Random().setstate((version, tuple(state), gauss))


def main():
    root = Path(sys.argv[1])
    name = sys.argv[2]
    results = root / "results/qwen"
    folder = results / "observations"
    raw = (results / "evaluation_manifest.json").read_bytes()
    manifest = json.loads(raw)
    manifest_hash = hashlib.sha256(raw).hexdigest()
    conditions = manifest["conditions"]
    order = [c["id"] for c in conditions]
    assert order == ["unchanged", "lora_128", "noise", "magnitude", "random"]
    index = order.index(name)
    assert index > 0
    assert len(manifest["probes"]) == len(set(manifest["probes"])) == 120
    assert digest(manifest["probes"]) == manifest["probes_sha256"]
    assert manifest["observations_per_probe_per_condition"] == 5000

    previous = json.loads((folder / "unchanged.json").read_text())
    verify_baseline(previous, manifest, manifest_hash)
    lengths = previous["prompt_lengths"]

    # Verify the saved chain before continuing its traffic randomization.
    for condition in conditions[1:index + 1]:
        name_in_chain = condition["id"]
        path = folder / f"{name_in_chain}.json"
        start_state = previous["python_random_state_after_sampling"]
        start_hash = digest(start_state)
        if path.exists():
            previous = json.loads(path.read_text())
            verify_changed(previous, condition, manifest, manifest_hash, start_hash, lengths)
            if name_in_chain == name:
                print(f"{name}: reused 600,000 verified observations.", flush=True)
                return
        else:
            assert name_in_chain == name, f"Run the missing condition first: {name_in_chain}"
            break

    bi = config.sampling.bi
    lp = config.sampling.logprob
    traffic = lp.other_prompts
    assert digest(traffic) == manifest["traffic_sha256"]
    assert (lp.batch_size, bi.per_batch_input_fraction) == (64, 0.95)
    assert bi.temperature == 0 and bi.models[manifest["model"]] == 1

    # Use the pinned baseline tokenizer for every condition, as in Step 13.
    tokenizer = AutoTokenizer.from_pretrained(conditions[0]["model_path"], local_files_only=True)
    patch_chat_template(tokenizer, config.chat_templates)
    model = AutoModelForCausalLM.from_pretrained(
        condition["model_path"], torch_dtype=torch.bfloat16, local_files_only=True,
    )
    assert get_model_hash(model) == condition["checkpoint_reloaded_hash"], name
    model = model.to("cuda:0")
    engine = init_vllm(model, tokenizer, "cuda:0")

    try:
        load_model_to_vllm(engine, model)
        dtype = engine.llm_engine.vllm_config.model_config.dtype
        assert dtype == torch.bfloat16
        engine_tokenizer = engine.get_tokenizer()
        for text in dict.fromkeys(manifest["probes"] + traffic):
            assert engine_tokenizer.encode(text) == tokenizer.encode(text), name

        restore_rng(start_state)
        print(f"Collecting {name}: 120 probes × 5,000 observations.", flush=True)
        sampled = vllm_inference_bi(
            llm=engine, inputs=manifest["probes"], traffic_prompts=traffic,
            n_samples=5000, temperature=bi.temperature,
            input_fraction=bi.per_batch_input_fraction,
            batch_size=lp.batch_size, max_tokens=1,
        )
        record = {
            "model": manifest["model"], "revision": manifest["revision"],
            "condition": name, "description": condition["description"],
            "checkpoint_hash": condition["checkpoint_reloaded_hash"],
            "manifest_sha256": manifest_hash, "inference_dtype": str(dtype),
            "start_rng_sha256": start_hash,
            "python_random_state_after_sampling": random.getstate(),
            "counts": sampled.counts, "prompt_lengths": sampled.prompt_lengths,
            "output_tokens": sampled.output_tokens,
        }
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(record, ensure_ascii=False))
        verify_changed(json.loads(tmp.read_text()), condition, manifest,
                       manifest_hash, start_hash, lengths)
        tmp.replace(path)
        print(f"{name}: 600,000 observations verified and saved to {path}", flush=True)
    finally:
        cleanup_vllm(engine)


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "TRACKLLM__SAMPLING__N_SAMPLES": "5000",
    "HF_HOME": str(root / "cache/huggingface"),
    "HF_HUB_OFFLINE": "1",
    "MPLBACKEND": "Agg",
}
started = time.monotonic()
for name in ("lora_128", "noise", "magnitude", "random"):
    log = root / f"logs/step14_qwen_{name}.log"
    with log.open("a", encoding="utf-8") as out:
        with subprocess.Popen(
            ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root), name],
            cwd=root / "authors_repo/track-llm-apis", env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                out.write(line)
                out.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f"{name} exited with code {status}. See {log}")

print("Step 14 passed: 3,000,000 fresh Qwen evaluation observations verified across five conditions.")
print("Saved in:", root / "results/qwen/observations")
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

2026-11-14 11:42:57,477 - track-llm-apis - INFO - Loading the lmsys/lmsys-chat-1m dataset...
2026-11-14 11:42:57,478 - track-llm-apis - INFO - Already processed dataset found at /workspace/B3IT_Figure1_Corrected/data/datasets/lmsys2flmsys-chat-1m, loading...
`torch_dtype` is deprecated! Use `dtype` instead!
Enforce eager set, overriding optimization level to -O0
The tokenizer you are loading from '/tmp/tmp646eua8a' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Allowing insecure serialization using pickle due to VLLM_ALLOW_INSECURE_SERIALIZATION=1
We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information.

In [ ]:
Step 15 — Prepare Qwen’s saved observations for the authors’ B3IT scoring code

In [22]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/prepare_qwen_scoring_input.py"

script.write_text(r'''
import gc
import hashlib
import json
import sys
import tempfile
from collections import Counter
from pathlib import Path

from track_llm_apis.sampling.common import CompressedOutput, DataSource
from track_llm_apis.tinychange import TinyChangeModel


def main():
    root = Path(sys.argv[1])
    results = root / "results/qwen"
    raw = (results / "evaluation_manifest.json").read_bytes()
    manifest = json.loads(raw)
    manifest_hash = hashlib.sha256(raw).hexdigest()
    probes = manifest["probes"]
    assert len(probes) == len(set(probes)) == 120
    assert [c["id"] for c in manifest["conditions"]] == [
        "unchanged", "lora_128", "noise", "magnitude", "random"
    ]

    destination = results / "figure1_merged"
    expected = Counter()
    records = []
    baseline_lengths = None

    for condition in manifest["conditions"]:
        name = condition["id"]
        path = results / "observations" / f"{name}.json"
        record = json.loads(path.read_text())
        assert record["manifest_sha256"] == manifest_hash, name
        assert record["model"] == manifest["model"] and record["revision"] == manifest["revision"], name
        assert record["condition"] == name and record["description"] == condition["description"], name
        assert record["inference_dtype"] == "torch.bfloat16" and record["output_tokens"] == 1, name
        if name != "unchanged":
            assert record["checkpoint_hash"] == condition["checkpoint_reloaded_hash"], name

        counts, lengths = record["counts"], record["prompt_lengths"]
        assert set(counts) == set(lengths) == set(probes), name
        if baseline_lengths is None:
            baseline_lengths = lengths
        assert lengths == baseline_lengths, name

        label = TinyChangeModel.description_to_str(condition["description"])
        for prompt, outputs in counts.items():
            assert all(type(n) is int and n > 0 for n in outputs.values()), (name, prompt)
            assert sum(outputs.values()) == 5000, (name, prompt)
            for text, n in outputs.items():
                expected[(label, (prompt, lengths[prompt]), (text, 1))] = n
        records.append((label, counts, lengths))
        print(f"{name}: 120 probes × 5,000 observations verified.", flush=True)

    assert sum(expected.values()) == 3_000_000

    def verify_archive(folder):
        loaded = CompressedOutput.from_json_dir(folder)
        assert loaded.model_name == manifest["model"]
        assert len(loaded.rows) == 3_000_000
        actual = Counter()
        for row in loaded.rows:
            assert row.source == DataSource.BI.value
            actual[(row.variant, row.prompt, row.text)] += 1
        assert actual == expected, "Reloaded archive differs from the observation files."

    if destination.exists():
        verify_archive(destination)
    else:
        with tempfile.TemporaryDirectory(dir=results, prefix=".qwen-merge-") as temp:
            staged = Path(temp) / "figure1_merged"
            merged = CompressedOutput(model_name=manifest["model"], gpus=["NVIDIA A40"])
            for label, counts, lengths in records:
                merged.add_bi_results(counts, lengths, variant=label, output_tokens=1)
            merged.dump_json(staged)
            del merged
            gc.collect()
            verify_archive(staged)
            staged.rename(destination)

    archive, = destination.glob("*.json.gz")
    print("Scoring input:", archive)
    print("Step 15 passed: 3,000,000 Qwen observations reload-verified in upstream format.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "CUDA_VISIBLE_DEVICES": "",
    "MPLBACKEND": "Agg",
    "HF_HUB_OFFLINE": "1",
}
started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root)],
    cwd=root / "authors_repo/track-llm-apis", env=env, check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

W1114 12:31:43.564000 8700 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


[2026-11-14 12:31:44,403] [WARNING] [real_accelerator.py:209:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
unchanged: 120 probes × 5,000 observations verified.
lora_128: 120 probes × 5,000 observations verified.
noise: 120 probes × 5,000 observations verified.
magnitude: 120 probes × 5,000 observations verified.
random: 120 probes × 5,000 observations verified.


2026-11-14 12:32:00,482 - track-llm-apis - INFO - Searching for a .json.gz file in /workspace/B3IT_Figure1_Corrected/results/qwen/.qwen-merge-sufomck4/figure1_merged...
2026-11-14 12:32:00,483 - track-llm-apis - INFO - Decompressing /workspace/B3IT_Figure1_Corrected/results/qwen/.qwen-merge-sufomck4/figure1_merged/Qwen2fQwen2.5-0.5B-Instruct.json.gz...
2026-11-14 12:32:00,484 - track-llm-apis - INFO - Loading JSON data from contents of the file...
2026-11-14 12:32:03,256 - track-llm-apis - INFO - Loading compressed output v1...


Scoring input: /workspace/B3IT_Figure1_Corrected/results/qwen/figure1_merged/Qwen2fQwen2.5-0.5B-Instruct.json.gz
Step 15 passed: 3,000,000 Qwen observations reload-verified in upstream format.
Elapsed: 0.7 minutes


In [ ]:
Step 16 — Run and verify the Qwen B3IT scoring sweep

In [23]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
repo = root / "authors_repo"
project = repo / "track-llm-apis"
script = root / "scripts/score_figure1_model.py"

# Apply a narrowly scoped, repeatable fix against the exact pinned source.
commit = "ad2f69295797a5f03e7b480246080d91087d9007"
relative = "track-llm-apis/src/track_llm_apis/sampling/analyze_logprobs.py"
original = subprocess.check_output(
    ["git", "show", f"{commit}:{relative}"], cwd=repo, text=True
)
old = "from track_llm_apis.analyze import ResponseData"
assert original.count(old) == 1
fixed = original.replace(old, "ResponseData = Any  # Legacy LT annotation; type absent at this commit.")
module = repo / relative
current = module.read_text()
assert current in (original, fixed), "This module has other edits. Share the output."
if current == original:
    module.write_text(fixed)
diff = subprocess.check_output(
    ["git", "diff", commit, "--", relative], cwd=repo, text=True
)
(root / "audit/step16_lt_import_compatibility.diff").write_text(diff)

script.write_text(r'''
import hashlib
import itertools
import json
import math
import random
import sys
import tempfile
from pathlib import Path

import numpy as np
import torch

from track_llm_apis.config import config
from track_llm_apis.sampling.analyze_param_sweeps import BIParameterSweep, compute_auc_from_stats
from track_llm_apis.sampling.common import (
    CompressedOutput, DataSource, get_analysis_dir, load_single_variant_results,
)
from track_llm_apis.tinychange import TinyChangeModel


def main():
    results = Path(sys.argv[1])
    manifest_file = results / "evaluation_manifest.json"
    manifest = json.loads(manifest_file.read_text())
    directory = results / "figure1_merged"
    archive, = directory.glob("*.json.gz")
    analysis = config.analysis
    analysis.device = "cpu"
    prompts = analysis.bi_n_prompts_values
    samples = analysis.bi_detection_samples_values
    reference = manifest["reference_samples_per_prompt"]
    assert prompts == [1, 3, 5, 10] and samples == [1, 3, 10, 20, 50]
    assert reference == 50 and analysis.n_tests == 1000
    assert analysis.param_chunk_index is None and config.seed == 0

    variants = [
        TinyChangeModel.description_to_str(c["description"])
        for c in manifest["conditions"] if c["id"] != "unchanged"
    ]
    expected = set(itertools.product(variants, prompts, samples))
    assert len(expected) == 80

    def verify(folder):
        rows = load_single_variant_results([folder], DataSource.BI)
        assert len(rows) == 80, f"Expected 80 result files, found {len(rows)}"
        aucs = {}
        for row in rows:
            assert row.model_name == manifest["model"] and row.method == DataSource.BI
            key = (row.variant, row.params["n_prompts"], row.params["detection_samples"])
            assert key in expected and key not in aucs, key
            h0, h1 = row.h0_stats.stats, row.h1_stats.stats
            assert row.n_tests == 1000 and (len(h0), len(h1)) == (2000, 1000), key
            assert all(math.isfinite(x) for x in itertools.chain(h0, h1)), key
            auc = float(compute_auc_from_stats(h0, h1, sampling=False))
            assert 0 <= auc <= 1, key
            aucs[key] = auc
        assert set(aucs) == expected
        return aucs

    inputs = {
        "archive_sha256": hashlib.sha256(archive.read_bytes()).hexdigest(),
        "manifest_sha256": hashlib.sha256(manifest_file.read_bytes()).hexdigest(),
        "seed": config.seed,
        "reference_samples": reference,
        "n_tests": analysis.n_tests,
        "device": analysis.device,
    }
    destination = get_analysis_dir(directory, DataSource.BI)
    marker = destination / "scoring_inputs.meta"
    if destination.exists():
        assert marker.is_file() and json.loads(marker.read_text()) == inputs, (
            f"Existing analysis has different or unverified provenance: {destination}"
        )
        aucs = verify(directory)
    else:
        data = CompressedOutput.from_json_dir(directory)
        assert data.model_name == manifest["model"] and len(data.rows) == 3_000_000
        random.seed(config.seed)
        np.random.seed(config.seed)
        torch.manual_seed(config.seed)

        with tempfile.TemporaryDirectory(dir=results, prefix=".scoring-") as tmp:
            stage = Path(tmp)
            BIParameterSweep.compute_stats(
                directory=stage, data=data,
                n_prompts_values=prompts, detection_samples_values=samples,
                reference_samples_per_prompt=reference, n_tests=analysis.n_tests,
            )
            aucs = verify(stage)
            staged_results = get_analysis_dir(stage, DataSource.BI)
            (staged_results / marker.name).write_text(json.dumps(inputs, indent=2))
            destination.parent.mkdir(parents=True, exist_ok=True)
            staged_results.rename(destination)

    print(f"Verified {len(aucs)} comparisons across {len(prompts) * len(samples)} settings.")
    print("Statistics per comparison: 2,000 H0 and 1,000 H1")
    print(f"Per-variant ROC-AUC range: {min(aucs.values()):.6f} to {max(aucs.values()):.6f}")
    print("Saved statistics:", destination)
    print("Step 16 passed: all 80 Qwen B3IT comparisons scored and verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

env = {
    **os.environ,
    "TRACKLLM__DATA_DIR": str(root / "data"),
    "CUDA_VISIBLE_DEVICES": "",
    "MPLBACKEND": "Agg",
    "HF_HUB_OFFLINE": "1",
}
started = time.monotonic()
log = root / "logs/step16_qwen_scoring.log"
with log.open("a", encoding="utf-8") as out:
    with subprocess.Popen(
        ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root / "results/qwen")],
        cwd=project, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            out.write(line)
            out.flush()
        status = process.wait()

print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")
if status:
    raise RuntimeError(f"Step 16 exited with code {status}. See {log}")

W1114 12:46:14.696000 8848 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'
[2026-11-14 12:46:15,482] [WARNING] [real_accelerator.py:209:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.
2026-11-14 12:46:17,805 - track-llm-apis - INFO - Searching for a .json.gz file in /workspace/B3IT_Figure1_Corrected/results/qwen/figure1_merged...
2026-11-14 12:46:17,806 - track-llm-apis - INFO - Decompressing /workspace/B3IT_Figure1_Corrected/results/qwen/figure1_merged/Qwen2fQwen2.5-0.5B-Instruct.json.gz...
2026-11-14 12:46:17,807 - track-llm-apis - INFO - Loading JSON data from contents of the file...
2026-11-14 12:46:20,701 - track-llm-apis - INFO - Loading compressed output v1...
2026-11-14 12:46:34,666 - track-llm-apis - INFO - B3IT: Building shared vocabulary across all variants...
2026-11-14 12:46:37,247 - track-llm-apis - INFO - B3IT: Precomputing counts for unchanged model...
2026-11-1

In [ ]:
Step 17 — Export and verify the Qwen Figure 1 CSVs

In [24]:
import os
import subprocess
import time
from pathlib import Path

root = Path("/workspace/B3IT_Figure1_Corrected")
script = root / "scripts/export_figure1_csvs.py"

script.write_text(r'''
import csv
import json
import math
import sys
from collections import defaultdict
from itertools import product
from pathlib import Path

from track_llm_apis.sampling.analyze_param_sweeps import (
    BIParameterSweep, compute_auc_from_stats,
)
from track_llm_apis.sampling.common import DataSource, load_single_variant_results
from track_llm_apis.tinychange import TinyChangeModel


def write_csv(path, rows):
    tmp = path.with_suffix(".csv.tmp")
    with tmp.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    with tmp.open(newline="", encoding="utf-8") as f:
        saved = list(csv.DictReader(f))
    assert saved == [{k: str(v) for k, v in row.items()} for row in rows]
    tmp.replace(path)


def main():
    folder = Path(sys.argv[1])
    manifest = json.loads((folder / "evaluation_manifest.json").read_text())
    model = manifest["model"]
    variants = [
        TinyChangeModel.description_to_str(c["description"])
        for c in manifest["conditions"] if c["id"] != "unchanged"
    ]
    grid = list(product([1, 3, 5, 10], [1, 3, 10, 20, 50]))
    assert len(set(variants)) == 4
    assert manifest["reference_samples_per_prompt"] == 50

    results = load_single_variant_results([folder / "figure1_merged"], DataSource.BI)
    indexed = {}
    for r in results:
        assert r.model_name == model and r.method == DataSource.BI
        key = (r.params["n_prompts"], r.params["detection_samples"], r.variant)
        assert key not in indexed, f"Duplicate comparison: {key}"
        assert r.n_tests == 1000
        assert len(r.h0_stats.stats) == 2000 and len(r.h1_stats.stats) == 1000
        indexed[key] = r
    assert set(indexed) == {(p, s, v) for p, s in grid for v in variants}

    per_variant, summary = [], []
    for p, s in grid:
        setting_rows = []
        input_tokens, output_tokens = [], []
        for v in variants:
            r = indexed[p, s, v]
            auc = float(compute_auc_from_stats(r.h0_stats.stats, r.h1_stats.stats, sampling=False))
            assert math.isfinite(auc) and 0 <= auc <= 1
            inp, out = r.h0_stats.input_token_avg, r.h0_stats.output_token_avg
            point = BIParameterSweep.create_result((p, s), inp, out)
            row = {
                "model": model, "variant": v, "n_prompts": p,
                "samples_per_prompt": s, "roc_auc": auc,
                "yearly_cost_usd": BIParameterSweep._compute_yearly_cost(point),
            }
            setting_rows.append(row)
            input_tokens.append(inp)
            output_tokens.append(out)

        per_variant.extend(setting_rows)
        # Upstream averages token counts across pairs before calculating cost.
        mean_point = BIParameterSweep.create_result(
            (p, s), sum(input_tokens) / 4, sum(output_tokens) / 4,
        )
        summary.append({
            "n_prompts": p, "samples_per_prompt": s,
            "mean_roc_auc": sum(row["roc_auc"] for row in setting_rows) / 4,
            "yearly_cost_usd": BIParameterSweep._compute_yearly_cost(mean_point),
            "number_of_variants": 4,
        })

    output = folder / "figure1"
    output.mkdir(exist_ok=True)
    for filename, rows in [
        ("figure1_per_variant.csv", per_variant),
        ("figure1_summary.csv", summary),
    ]:
        path = output / filename
        write_csv(path, rows)
        print(f"{len(rows)} rows saved and reread: {path}")

    for row in (summary[0], summary[-1]):
        print(f'{row["n_prompts"]} prompt(s) × {row["samples_per_prompt"]} detection '
              f'sample(s): mean AUC {row["mean_roc_auc"]:.6f}, '
              f'annual cost ${row["yearly_cost_usd"]:.4f}')
    print("Step 17 passed: 80 per-variant rows and 20 four-variant means verified.")


if __name__ == "__main__":
    main()
''', encoding="utf-8")

started = time.monotonic()
subprocess.run(
    ["/opt/b3it-corrected-venv/bin/python", "-u", str(script), str(root / "results/qwen")],
    cwd=root / "authors_repo/track-llm-apis",
    env={
        **os.environ,
        "TRACKLLM__DATA_DIR": str(root / "data"),
        "CUDA_VISIBLE_DEVICES": "",
        "MPLBACKEND": "Agg",
        "HF_HUB_OFFLINE": "1",
    },
    check=True,
)
print(f"Elapsed: {(time.monotonic() - started) / 60:.1f} minutes")

W1114 13:02:23.138000 9053 torch/utils/cpp_extension.py:117] No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


[2026-11-14 13:02:23,834] [WARNING] [real_accelerator.py:209:get_accelerator] Setting accelerator to CPU. If you have GPU or other accelerator, we were unable to detect it.


2026-11-14 13:02:26,123 - track-llm-apis - INFO - [DataSource.BI] Loading results from /workspace/B3IT_Figure1_Corrected/results/qwen/figure1_merged...


80 rows saved and reread: /workspace/B3IT_Figure1_Corrected/results/qwen/figure1/figure1_per_variant.csv
20 rows saved and reread: /workspace/B3IT_Figure1_Corrected/results/qwen/figure1/figure1_summary.csv
1 prompt(s) × 1 detection sample(s): mean AUC 0.699689, annual cost $0.1314
10 prompt(s) × 50 detection sample(s): mean AUC 0.936773, annual cost $65.7000
Step 17 passed: 80 per-variant rows and 20 four-variant means verified.
Elapsed: 0.2 minutes
